### Setting up Imports

In [ ]:
import datasets
import os
from itertools import islice
from dotenv import load_dotenv
load_dotenv()
datasets.logging.set_verbosity_error()
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)
from common.helper import clickhouse_connect,clickhouse_extract_data_as_df
from common.const import TABLE_RAW_PRODUCTS,TABLE_RAW_REVIEWS

### Connecting to ClickHouse and Getting Data for Product-Product Similarity

In [8]:
SCHEMA = "amazon"
# Products
products_query = f"""
SELECT
    parent_asin,
    title,
    description,
    features,
    main_category,
    details,
    CAST(price AS Nullable(Float64)) AS price,
    average_rating,
    rating_number
FROM {SCHEMA}.{TABLE_RAW_PRODUCTS}
LIMIT 450000 OFFSET 50000
"""
products_df = clickhouse_extract_data_as_df(products_query)

connected to clickhouse


### Feature Engineering

In [9]:
import pandas as pd
def feature_engineering(products_df:pd.DataFrame):

    df = products_df.fillna('')
    
    products_df['combined_text'] = (
        (df['title'] + " ") * 1 + df['title'] + " " +
        df['description'] + " " +
        df['features'] + " " +
        (df['main_category'] + " ") * 1 + df['main_category'] + " " +
        df['details']
    ).str.strip()

    return products_df

In [10]:
products_df = feature_engineering(products_df)

In [11]:
products_df['combined_text'][0]

'Advanced Squad Leader Starter Kit #1 Advanced Squad Leader Starter Kit #1   Toys & Games Toys & Games {"Manufacturer": "Kelllelldaaa"}'

### Pinecone Creating a NameSpace

In [ ]:
from pinecone import Pinecone, ServerlessSpec

###hide this later
pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])
print(pc)

### Vectorization and Upserting Records in Batches

#### Vectorization

In [13]:
import torch
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(DEVICE)

cuda:0


In [14]:
from sentence_transformers import SentenceTransformer
def vectorize(products_df):
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

    # Generate embeddings (384 dimensions)
    semantic_embeddings = model.encode(
        products_df['combined_text'].tolist(),
        show_progress_bar=True,
        batch_size=128,
        normalize_embeddings=True,  # Normalize for cosine similarity
        device=DEVICE
    )
    return semantic_embeddings

#### Upserting

In [15]:
def upsertinBatches(index,products_df,semantic_embeddings,num_batches=200):
    import numpy as np
    data_for_upsert = []
    for i, row in products_df.iterrows():
        data_for_upsert.append({
            "id": str(row['parent_asin']), 
            "values": semantic_embeddings[i].tolist(), 
            "metadata": {
                "title": str(row['title'])[:500], # Metadata limit: truncate if too long
                "main_category": str(row['main_category'])
            }
        })
    
    for i in range(0, len(data_for_upsert), num_batches):
    
        batch = data_for_upsert[i : i + num_batches]
        
        index.upsert(vectors=batch, namespace='reviews')
        
        if (i + num_batches) % 1000 == 0 or (i + num_batches) >= len(data_for_upsert):
            print(f"Upserted {min(i + num_batches, len(data_for_upsert))} / {len(data_for_upsert)} vectors")
    
    print("Success: All batches uploaded to Pinecone.")


#### Connecting to Pinecone and calling Functions

In [16]:
index = pc.Index(host="https://scc454-custom-hlhdcpk.svc.aped-4627-b74a.pinecone.io")
index

In [17]:
# generate embeddings
embeddings = vectorize(products_df)

#upsert
upsertinBatches(index=index,products_df=products_df,semantic_embeddings=embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/3516 [00:00<?, ?it/s]

Upserted 1000 / 450000 vectors
Upserted 2000 / 450000 vectors
Upserted 3000 / 450000 vectors
Upserted 4000 / 450000 vectors
Upserted 5000 / 450000 vectors
Upserted 6000 / 450000 vectors
Upserted 7000 / 450000 vectors
Upserted 8000 / 450000 vectors
Upserted 9000 / 450000 vectors
Upserted 10000 / 450000 vectors
Upserted 11000 / 450000 vectors
Upserted 12000 / 450000 vectors
Upserted 13000 / 450000 vectors
Upserted 14000 / 450000 vectors
Upserted 15000 / 450000 vectors
Upserted 16000 / 450000 vectors
Upserted 17000 / 450000 vectors
Upserted 18000 / 450000 vectors
Upserted 19000 / 450000 vectors
Upserted 20000 / 450000 vectors
Upserted 21000 / 450000 vectors
Upserted 22000 / 450000 vectors
Upserted 23000 / 450000 vectors
Upserted 24000 / 450000 vectors
Upserted 25000 / 450000 vectors
Upserted 26000 / 450000 vectors
Upserted 27000 / 450000 vectors
Upserted 28000 / 450000 vectors
Upserted 29000 / 450000 vectors
Upserted 30000 / 450000 vectors
Upserted 31000 / 450000 vectors
Upserted 32000 / 

### Getting Data for User-User Similarity

In [18]:

# Reviews
reviews_query = f"""
SELECT 
    user_id,
    parent_asin,
    title,
    text,
    rating
FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
LIMIT 450000 OFFSET 50000
"""
reviews_df = clickhouse_extract_data_as_df(reviews_query)

connected to clickhouse


### User Data Feature Engineering

In [19]:
reviews_df['full_review'] = (
    reviews_df['title'].fillna('') + ' ' +
    reviews_df['text'].fillna('')
)

user_reviews = (
    reviews_df
    .groupby('user_id')['full_review']
    .apply(' '.join)
    .reset_index(name='all_reviews_text')
)

In [20]:
user_reviews

,user_id,all_reviews_text
0,AE222ABE7SFNVT34H5XDXASHAP5A,This is a fun game! I enjoy playing this game ...
1,AE222H3FGXWLHRFUMGMS2RR57NDQ,Grandchildren love it ! Grandchildren love it ...
2,AE222Y4WTST6BUZ4J5Y2H6QMBITQ,Lensoclean Unit The cleaning unit does a good ...
3,AE2233IFF7ELQDTUC4HVGODPL6KA,The best fast pace laugh ever! We got the Hoop...
4,AE2236PYACSGGLZLHHUWKGESHYTA,very happy This set is perfect. It has all the...
...,...,...
378433,AHZZYSBK2XCQQIGSHM5LUCFJER5A,You would think the thick pieces are a good th...
378434,AHZZZ6OLGGSAUFFBL2W5FRPVVPPQ,Pricey! Small. Not big enough to cover the ......
378435,AHZZZMHL52TKKPQESWCFCMSXXBMA,Three Stars It works but the quality is ok. Th...
378436,AHZZZMZFYIFIAYVN6O34AOMKYFNQ,Five Stars A+


### User Data Vectorization and Upserting

In [21]:
from sentence_transformers import SentenceTransformer
def vectorize2(user_df):
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

    # Generate embeddings (384 dimensions)
    semantic_embeddings = model.encode(
        user_df['all_reviews_text'].tolist(),
        show_progress_bar=True,
        batch_size=32,
        normalize_embeddings=True,  # Normalize for cosine similarity
        device=DEVICE
    )
    return semantic_embeddings

In [22]:
def upsertinBatches2(index,products_df,semantic_embeddings,num_batches=200):
    import numpy as np
    data_for_upsert = []
    for i, row in products_df.iterrows():
        data_for_upsert.append({
            "id": str(row['user_id']), 
            "values": semantic_embeddings[i].tolist()
        })
    
    for i in range(0, len(data_for_upsert), num_batches):
    
        batch = data_for_upsert[i : i + num_batches]
        
        index.upsert(vectors=batch, namespace='user')
        
        if (i + num_batches) % 1000 == 0 or (i + num_batches) >= len(data_for_upsert):
            print(f"Upserted {min(i + num_batches, len(data_for_upsert))} / {len(data_for_upsert)} vectors")
    
    print("Success: All batches uploaded to Pinecone.")


In [23]:
# generate embeddings
user_embeddings = vectorize2(user_df=user_reviews)

#upsert
upsertinBatches2(index=index,products_df=user_reviews,semantic_embeddings=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/11827 [00:00<?, ?it/s]

Upserted 1000 / 378438 vectors
Upserted 2000 / 378438 vectors
Upserted 3000 / 378438 vectors
Upserted 4000 / 378438 vectors
Upserted 5000 / 378438 vectors
Upserted 6000 / 378438 vectors
Upserted 7000 / 378438 vectors
Upserted 8000 / 378438 vectors
Upserted 9000 / 378438 vectors
Upserted 10000 / 378438 vectors
Upserted 11000 / 378438 vectors
Upserted 12000 / 378438 vectors
Upserted 13000 / 378438 vectors
Upserted 14000 / 378438 vectors
Upserted 15000 / 378438 vectors
Upserted 16000 / 378438 vectors
Upserted 17000 / 378438 vectors
Upserted 18000 / 378438 vectors
Upserted 19000 / 378438 vectors
Upserted 20000 / 378438 vectors
Upserted 21000 / 378438 vectors
Upserted 22000 / 378438 vectors
Upserted 23000 / 378438 vectors
Upserted 24000 / 378438 vectors
Upserted 25000 / 378438 vectors
Upserted 26000 / 378438 vectors
Upserted 27000 / 378438 vectors
Upserted 28000 / 378438 vectors
Upserted 29000 / 378438 vectors
Upserted 30000 / 378438 vectors
Upserted 31000 / 378438 vectors
Upserted 32000 / 

### Vector Search Over Products

#### Defining Products to get

In [24]:
random_products = products_df['parent_asin'].sample(n=10).tolist()
random_products

['B00UHE2SFO',
 'B007WZKTUW',
 'B006YYPMC2',
 'B08DJ4RW5L',
 'B0047WOQZC',
 'B0042H6O98',
 'B001H02MDK',
 'B00HQ5KJZI',
 'B00YAGVWIO',
 'B00VQ08MS0']

##### since Pinecone doesn't support KNN we build a ground truth using FAISS flat-indexing

In [25]:
import faiss
import numpy as np

def get_ground_truth_results(embeddings, query_vector, k=10):
    d = embeddings.shape[1]
    xb = np.ascontiguousarray(embeddings).astype('float32')
    index = faiss.IndexFlatL2(d)
    index.add(xb)
    
    xq = np.array([query_vector]).astype('float32')
    distances, indices = index.search(xq, k)
    
    return indices[0], distances[0]

#### Getting the Exact Matches and Pinecone Similarity and Calculating Precision

In [26]:
import numpy as np
import time

# 1. Setup FAISS once outside to ensure we only time the SEARCH
d = embeddings.shape[1]
xb = np.ascontiguousarray(embeddings).astype('float32')
faiss_index = faiss.IndexFlatL2(d)
faiss_index.add(xb)

precision_scores = []
pinecone_latencies = []
exact_latencies = []

for product_id in random_products:
    # Get vector for query
    result_df = products_df[products_df['parent_asin'] == product_id]
    if result_df.empty: continue
    query_vector = vectorize(result_df)[0]
    xq = np.array([query_vector]).astype('float32')
    
    # --- TIME EXACT SEARCH (FAISS) ---
    start_exact = time.time()
    D_exact, I_exact = faiss_index.search(xq, 11) # top_k=11
    end_exact = time.time()
    
    exact_latencies.append((end_exact - start_exact) * 1000)
    ground_truth_ids = set(products_df.iloc[I_exact[0]]['parent_asin'].astype(str).tolist())

    # --- TIME APPROXIMATE SEARCH (PINECONE) ---
    start_pine = time.time()
    pinecone_results = index.query(
        id=product_id,
        top_k=11,
        namespace='reviews'
    )
    end_pine = time.time()
    
    pinecone_latencies.append((end_pine - start_pine) * 1000)
    pinecone_ids = set([str(match['id']) for match in pinecone_results['matches']])
    
    # --- ACCURACY ---
    correct_matches = ground_truth_ids.intersection(pinecone_ids)
    precision_scores.append(len(correct_matches) / 11)

# --- STATISTICAL HELPER FUNCTION ---
def print_stats(name, data):
    mu = np.mean(data)
    sigma = np.std(data)
    print(f"{name} Mean: {mu:.4f}")
    print(f"{name} Std Dev: {sigma:.4f}")
    print(f"{name} Range (±2σ): [{max(0, mu - 2*sigma):.2f}, {mu + 2*sigma:.2f}]")
    print("-" * 30)

# --- RESULTS ---
print("--- ACCURACY REPORT ---")
print_stats("Precision@11", precision_scores)

print("--- LATENCY REPORT (ms) ---")
print_stats("Pinecone Latency", pinecone_latencies)
print_stats("Exact Latency", exact_latencies)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

--- ACCURACY REPORT ---
Precision@11 Mean: 0.9636
Precision@11 Std Dev: 0.0603
Precision@11 Range (±2σ): [0.84, 1.08]
------------------------------
--- LATENCY REPORT (ms) ---
Pinecone Latency Mean: 230.7647
Pinecone Latency Std Dev: 40.0746
Pinecone Latency Range (±2σ): [150.62, 310.91]
------------------------------
Exact Latency Mean: 69.2292
Exact Latency Std Dev: 4.7235
Exact Latency Range (±2σ): [59.78, 78.68]
------------------------------


### Vector Search for Users

In [27]:
random_users = user_reviews['user_id'].sample(n=5).tolist()
random_users

['AH66VKOND3WMW4J4KJBY72DEWSCA',
 'AE2GEZWTMO4MFEIDGTYU2QQSSNVA',
 'AHROYKYM4WV2QDJIAHMB2S2XEL6Q',
 'AES5LPUIOARAFFMYSHMMISINE7JA',
 'AF5N5XVTGNE3GPDRJMR6HYX6LENQ']

In [28]:
import numpy as np
import time

precision_scores = []
pinecone_latencies = []
exact_latencies = []

# Assuming user_embeddings and faiss_index are already defined/built
for user_id in random_users:
    result_df = user_reviews[user_reviews['user_id'] == user_id]
    if result_df.empty: continue
    query_vector = vectorize2(result_df)[0]
    
    # --- TIME EXACT SEARCH (FAISS) ---
    start_exact = time.time()
    I, D = get_ground_truth_results(embeddings=user_embeddings, query_vector=query_vector)
    end_exact = time.time()
    exact_latencies.append((end_exact - start_exact) * 1000)
    
    # Convert IDs to strings to ensure matching works
    ground_truth_ids = set(user_reviews.iloc[I]['user_id'].astype(str).tolist())

    # --- TIME APPROXIMATE SEARCH (PINECONE) ---
    start_pine = time.time()
    pinecone_results = index.query(
        id=user_id,
        top_k=11,
        include_values=False,
        include_metadata=True,
        namespace='user'
    )
    end_pine = time.time()
    pinecone_latencies.append((end_pine - start_pine) * 1000)

    pinecone_ids = set([str(match['id']) for match in pinecone_results['matches']])
    
    # --- METRICS ---
    correct_matches = ground_truth_ids.intersection(pinecone_ids)
    precision = len(correct_matches) / 11
    precision_scores.append(precision)

# --- STATISTICAL REPORTING ---
def print_full_stats(label, data, unit=""):
    mean_val = np.mean(data)
    std_val = np.std(data)
    low = mean_val - 2 * std_val
    high = mean_val + 2 * std_val
    # Clamp bounds for precision (0 to 1)
    if unit == "": 
        low, high = max(0, low), min(1, high)
        
    print(f"{label} Mean: {mean_val:.4f}{unit}")
    print(f"{label} Std Dev: {std_val:.4f}")
    print(f"{label} Range (±2σ): [{low:.4f}{unit}, {high:.4f}{unit}]")
    print("-" * 30)

print("--- PERFORMANCE & ACCURACY REPORT ---")
print_full_stats("Precision@11", precision_scores)
print_full_stats("Pinecone Latency", pinecone_latencies, " ms")
print_full_stats("Exact Latency", exact_latencies, " ms")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

--- PERFORMANCE & ACCURACY REPORT ---
Precision@11 Mean: 0.0909
Precision@11 Std Dev: 0.0000
Precision@11 Range (±2σ): [0.0909, 0.0909]
------------------------------
Pinecone Latency Mean: 207.8844 ms
Pinecone Latency Std Dev: 47.6655
Pinecone Latency Range (±2σ): [112.5534 ms, 303.2153 ms]
------------------------------
Exact Latency Mean: 684.4021 ms
Exact Latency Std Dev: 12.4196
Exact Latency Range (±2σ): [659.5629 ms, 709.2412 ms]
------------------------------
